# PDF PII Redaction with Presidio

**Combines:**
- Document Intelligence JSON extraction (raw coordinates)
- Presidio financial PII detection & text masking
- PyMuPDF black-box PDF redaction

**Input:** PDF + raw extraction JSON  
**Output:** Masked text + redacted PDF

## Step 1: Setup & Install Dependencies

In [ ]:
!pip install -q pymupdf presidio-analyzer presidio-anonymizer spacy
!python -m spacy download -q en_core_web_sm
print("✓ Dependencies installed")

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)
print("✓ Google Drive mounted at /content/gdrive")

## Step 2: Import Core Classes

In [ ]:
import json
import re
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import fitz  # PyMuPDF
from functools import partial

from presidio_analyzer import AnalyzerEngine, Pattern, PatternRecognizer
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

print("✓ Core imports successful")

## Step 3: Financial PII Masker (Presidio-based)

In [ ]:
class FinancialPIIMasker:
    """PII masking using Presidio for financial documents."""
    
    _BUILTIN_ENTITIES = [
        "PERSON", "PHONE_NUMBER", "EMAIL_ADDRESS", "LOCATION", "DATE_TIME",
        "CREDIT_CARD", "IBAN_CODE", "US_SSN", "US_PASSPORT",
        "US_BANK_NUMBER", "CRYPTO",
    ]

    _CUSTOM_PATTERNS = [
        ("CUSTOMER_ID", r"\bCUST-\d{6,10}\b", 0.9),
        ("CVV", r"\bCVV[:\s=]*\d{3,4}\b", 0.85),
        ("UK_NINO", r"(?i)(?<![A-Za-z0-9])(?!GB|NK|TN|ZZ)[A-Z]{2} ?\d{2} ?\d{2} ?\d{2} ?[A-D]\b", 0.85),
        ("UK_SORT_CODE", r"(?i)\bsort code[:=]?\s*\d{2}-\d{2}-\d{2}\b", 0.85),
        ("LABELED_ACCOUNT", r"(?i)\b(?:account|acct)\s*(?:number|no\.?|num)\s*(?:is|was)?\s*[:=]?\s*\d{6,12}\b", 0.85),
        ("UK_DRIVER_LICENSE", r"\b[A-Z9]{5}\d{6}[A-Z9]{2}[0-9A-Z]{3}\b", 0.75),
        ("UK_POSTCODE", r"(?i)\b(?:GIR ?0AA|(?:[A-Z]{1,2}[0-9][A-Z0-9]?) ?[0-9][A-Z]{2})\b", 0.85),
        ("INTL_PHONE", r"(?<![\w+])\+\d{1,3}[\s-]?\(?\d[\d\s\-()]{6,16}\d", 0.6),
    ]

    _analyzer: Optional[AnalyzerEngine] = None
    _anonymizer: Optional[AnonymizerEngine] = None

    def __init__(self, text: str, placeholder_style: str = "typed"):
        self._original = text
        self._masked: Optional[str] = None
        self._placeholder_style = placeholder_style
        
        if FinancialPIIMasker._analyzer is None:
            FinancialPIIMasker._analyzer = self._build_analyzer()
        if FinancialPIIMasker._anonymizer is None:
            FinancialPIIMasker._anonymizer = AnonymizerEngine()

    @classmethod
    def _build_analyzer(cls):
        analyzer = AnalyzerEngine()
        for entity, regex, score in cls._CUSTOM_PATTERNS:
            analyzer.registry.add_recognizer(
                PatternRecognizer(
                    supported_entity=entity,
                    patterns=[Pattern(name=f"{entity.lower()}_pattern", regex=regex, score=score)],
                )
            )
        return analyzer

    def placeholder_for(self, entity: str) -> str:
        if self._placeholder_style == "typed":
            return f"<{entity}>"
        return "<REDACTED>"

    def mask_data(self) -> str:
        entities = self._BUILTIN_ENTITIES + [p[0] for p in self._CUSTOM_PATTERNS]
        results = self._analyzer.analyze(text=self._original, entities=entities, language="en")
        
        operators = {}
        for entity in entities:
            operators[entity] = OperatorConfig("replace", {"new_value": self.placeholder_for(entity)})
        
        anonymized = self._anonymizer.anonymize(
            text=self._original,
            analyzer_results=results,
            operators=operators,
        )
        self._masked = anonymized.text
        return self._masked

    def original_data(self) -> str:
        return self._original

print("✓ FinancialPIIMasker ready")

## Step 4: PDF Redaction Engine

In [ ]:
class PDFRedactionEngine:
    """Handle PDF redaction using coordinates from document extraction."""
    
    def __init__(self, pdf_path: str):
        self.pdf_path = Path(pdf_path)
        if not self.pdf_path.exists():
            raise FileNotFoundError(f"PDF not found: {self.pdf_path}")

    def redact_by_text_matches(self, extraction_json: Dict, sensitive_values: List[str], 
                                padding_pt: float = 1.5) -> Path:
        """Redact by finding text matches in extraction and mapping to coordinates."""
        content = extraction_json.get('content', '')
        pages = extraction_json.get('pages', [])
        redactions = []
        
        for page_info in pages:
            page_num = page_info['pageNumber']
            words = page_info.get('words', [])
            
            for sensitive_value in sensitive_values:
                matches = self._find_text_in_page(content, words, page_num, sensitive_value)
                redactions.extend(matches)
        
        return self._apply_redactions(redactions, padding_pt)

    def _find_text_in_page(self, content: str, words: List[Dict], page_num: int, 
                           search_value: str) -> List[Dict]:
        """Find text matches and convert to bounding boxes."""
        matches = []
        word_offsets = {}
        
        for word in words:
            if 'span' in word:
                offset = word['span']['offset']
                word_offsets[offset] = word
        
        if not word_offsets:
            return matches
        
        page_start = min(o for o in word_offsets.keys())
        page_end = max(o + word_offsets[o]['span']['length'] for o in word_offsets.keys())
        
        search_pos = page_start
        while True:
            idx = content.find(search_value, search_pos, page_end)
            if idx < 0:
                break
            
            end_idx = idx + len(search_value)
            matched_words = []
            
            for offset, word in word_offsets.items():
                word_end = offset + word['span']['length']
                if offset < end_idx and word_end > idx:
                    matched_words.append(word)
            
            if matched_words:
                bbox = self._combine_word_boxes(matched_words)
                if bbox:
                    matches.append({
                        'page': page_num,
                        'bbox_inches': bbox,
                        'value': search_value,
                    })
            
            search_pos = idx + max(1, len(search_value))
        
        return matches

    def _apply_redactions(self, redactions: List[Dict], padding_pt: float = 1.5) -> Path:
        """Apply redactions to PDF."""
        doc = fitz.open(self.pdf_path)
        
        for redaction in redactions:
            page_num = redaction['page'] - 1
            if page_num >= len(doc):
                continue
            
            page = doc[page_num]
            bbox = redaction['bbox_inches']
            x0, y0, x1, y1 = bbox
            rect = fitz.Rect(x0 * 72, y0 * 72, x1 * 72, y1 * 72)
            
            rect = fitz.Rect(
                rect.x0 - padding_pt,
                rect.y0 - padding_pt,
                rect.x1 + padding_pt,
                rect.y1 + padding_pt,
            )
            page.add_redact_annot(rect, fill=(0, 0, 0))
        
        for page in doc:
            page.apply_redactions()
        
        output_path = self.pdf_path.parent / f"{self.pdf_path.stem}_redacted.pdf"
        doc.save(output_path, garbage=4, deflate=True)
        doc.close()
        
        return output_path

    @staticmethod
    def _polygon_to_bbox(polygon: List[float]) -> Tuple[float, float, float, float]:
        xs = polygon[0::2]
        ys = polygon[1::2]
        return min(xs), min(ys), max(xs), max(ys)

    @staticmethod
    def _combine_word_boxes(words: List[Dict]) -> Optional[Tuple]:
        boxes = []
        for word in words:
            if word.get('polygon'):
                boxes.append(PDFRedactionEngine._polygon_to_bbox(word['polygon']))
        
        if not boxes:
            return None
        
        return (
            min(b[0] for b in boxes),
            min(b[1] for b in boxes),
            max(b[2] for b in boxes),
            max(b[3] for b in boxes),
        )

print("✓ PDFRedactionEngine ready")

## Step 5: Unified Redaction Pipeline

In [ ]:
class PIIRedactionPipeline:
    """Combined pipeline for text masking and PDF redaction."""
    
    def __init__(self, pdf_path: str, extraction_json: Dict):
        self.pdf_path = Path(pdf_path)
        self.extraction_json = extraction_json
        self.content = extraction_json.get('content', '')
        self.redaction_engine = PDFRedactionEngine(str(pdf_path))

    def process(self, sensitive_values: Optional[List[str]] = None, 
                use_presidio: bool = True) -> Dict:
        """
        Run full redaction pipeline.
        
        Args:
            sensitive_values: List of known sensitive strings to redact
            use_presidio: Whether to use Presidio for automatic PII detection
        
        Returns:
            Dict with masked_text, redacted_pdf_path, detections
        """
        # Step 1: Text masking with Presidio
        masker = FinancialPIIMasker(self.content)
        masked_text = masker.mask_data()
        
        # Step 2: Detect PII for PDF redaction
        detections = []
        
        if use_presidio:
            analyzer = FinancialPIIMasker._analyzer
            entities = FinancialPIIMasker._BUILTIN_ENTITIES + [p[0] for p in FinancialPIIMasker._CUSTOM_PATTERNS]
            results = analyzer.analyze(text=self.content, entities=entities, language="en")
            
            for result in results:
                detected_text = self.content[result.start:result.end]
                detections.append({
                    'value': detected_text,
                    'entity_type': result.entity_type,
                    'confidence': result.score,
                })
        
        if sensitive_values:
            detections.extend([{'value': v, 'entity_type': 'CUSTOM', 'confidence': 1.0} 
                              for v in sensitive_values])
        
        # Step 3: Apply PDF redactions
        all_values = [d['value'] for d in detections] + (sensitive_values or [])
        all_values = list(set(all_values))
        
        redacted_pdf_path = self.redaction_engine.redact_by_text_matches(
            self.extraction_json, all_values
        )
        
        return {
            'masked_text': masked_text,
            'original_text': self.content,
            'redacted_pdf_path': str(redacted_pdf_path),
            'detections': detections,
            'num_redactions': len(all_values),
        }

print("✓ PIIRedactionPipeline ready")

## Step 6: Upload Files

In [ ]:
from google.colab import files

print("Upload your extraction.json file...")
uploaded_json = files.upload()
json_file = list(uploaded_json.keys())[0]

with open(json_file, 'r') as f:
    extraction_json = json.load(f)

print(f"✓ Loaded {json_file}")
print(f"  Content length: {len(extraction_json.get('content', ''))} chars")
print(f"  Pages: {len(extraction_json.get('pages', []))}")

In [ ]:
print("Upload your PDF file...")
uploaded_pdf = files.upload()
pdf_file = list(uploaded_pdf.keys())[0]

print(f"✓ Loaded {pdf_file}")
doc = fitz.open(pdf_file)
print(f"  Pages: {len(doc)}")
doc.close()

## Step 7: Configuration

In [ ]:
# Define sensitive values to redact (in addition to automatic Presidio detection)
SENSITIVE_VALUES = [
    'ABC DEFGH',      # Name from the example
    '22-Sep-2025',    # Date from the example
    # Add more as needed
]

# PDF redaction padding (points = 1/72 inch)
REDACTION_PADDING_PT = 1.5

print("Configuration:")
print(f"  Sensitive values to redact: {len(SENSITIVE_VALUES)}")
print(f"  Redaction padding: {REDACTION_PADDING_PT} points")
print(f"  Use Presidio auto-detection: True")

## Step 8: Execute Redaction Pipeline

In [ ]:
# Initialize pipeline
pipeline = PIIRedactionPipeline(pdf_file, extraction_json)

# Run redaction
print("\n🔄 Running redaction pipeline...\n")
result = pipeline.process(sensitive_values=SENSITIVE_VALUES, use_presidio=True)

print("✓ Redaction complete!")
print(f"\n📊 Results:")
print(f"  Detections found: {len(result['detections'])}")
print(f"  Unique values redacted: {result['num_redactions']}")
print(f"  Redacted PDF: {Path(result['redacted_pdf_path']).name}")

## Step 9: View Results

In [ ]:
print("\n🔍 PII Detections:\n")
for i, detection in enumerate(result['detections'], 1):
    print(f"{i}. [{detection['entity_type']}] {detection['value'][:50]}... (confidence: {detection['confidence']:.2f})")

In [ ]:
print("\n📝 Original vs Masked Text (first 500 chars):\n")
print("ORIGINAL:")
print("-" * 80)
print(result['original_text'][:500])
print("\n" + "=" * 80)
print("\nMAKED:")
print("-" * 80)
print(result['masked_text'][:500])

## Step 10: Download Results

In [ ]:
# Save masked text
masked_text_file = Path(pdf_file).stem + "_masked.txt"
with open(masked_text_file, 'w') as f:
    f.write(result['masked_text'])

print(f"✓ Saved masked text to {masked_text_file}")

# Save detections JSON
detections_file = Path(pdf_file).stem + "_detections.json"
with open(detections_file, 'w') as f:
    json.dump(result['detections'], f, indent=2)

print(f"✓ Saved detections to {detections_file}")

In [ ]:
# Download redacted PDF
redacted_pdf = result['redacted_pdf_path']
files.download(redacted_pdf)
print(f"✓ Downloaded {Path(redacted_pdf).name}")

# Download masked text
files.download(masked_text_file)
print(f"✓ Downloaded {masked_text_file}")

# Download detections
files.download(detections_file)
print(f"✓ Downloaded {detections_file}")

## Summary

**What happened:**
1. ✅ Loaded Document Intelligence JSON with raw word coordinates
2. ✅ Detected PII using Presidio (financial entities + custom patterns)
3. ✅ Masked text by replacing detected values with `<ENTITY_TYPE>` placeholders
4. ✅ Mapped masked values to PDF coordinates from the extraction
5. ✅ Applied black-box redactions to the PDF
6. ✅ Exported masked text, redactions list, and redacted PDF

**Files created:**
- `*_redacted.pdf` - PDF with black-box redactions
- `*_masked.txt` - Text with PII replaced by entity labels
- `*_detections.json` - List of detected PII with confidence scores